# MLflow GenAI: Walkthrough & Evaluation

One notebook for **MLflow 3 GenAI** on Databricks: tracking, tracing, and evaluation. Combines setup, traced model calls, and pulling inference data into MLflow for evaluation.

**Docs:** [MLflow 3 for Generative AI](https://docs.databricks.com/aws/en/mlflow3/genai/)

In [ ]:
%pip install -U --quiet mlflow
# %restart_python

In [ ]:
import mlflow
import mlflow.deployments
import time

## 1. Tracking & registry

On Databricks, tracking and registry URIs are set automatically.

In [ ]:
print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())
print("MLflow version:", mlflow.__version__)

## 2. Deploy client & traced model call

Use the deploy client to call a foundation model. `@mlflow.trace` records the call so you can inspect latency and inputs/outputs in the MLflow UI.

In [ ]:
client = mlflow.deployments.get_deploy_client("databricks")

# List endpoints (optional)
endpoints = client.list_endpoints()
for ep in endpoints[:5]:
    print("-", ep.get("name", ep))

In [ ]:
MODEL_ENDPOINT = "databricks-llama-4-maverick"  # or another endpoint you have

@mlflow.trace
def query_model(prompt: str, temperature: float = 0.7, max_tokens: int = 100):
    response = client.predict(
        endpoint=MODEL_ENDPOINT,
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens,
        },
    )
    return response["choices"][0]["message"]["content"]

## 3. Set experiment and run

One experiment per app is recommended. Enable autolog so traces are recorded.

In [ ]:
# Use your username or a shared path
experiment_name = "/Users/your_user/mlflow_genai_evaluation"
mlflow.set_experiment(experiment_name)

mlflow.autolog()
out = query_model("Quote Marcus Aurelius in one sentence.", temperature=0.5)
print(out)

## 4. Evaluation: inference table → MLflow

If you have an **inference table** (e.g. from model serving or logged payloads), load it and log runs or link traces to MLflow for evaluation. Replace the table name with your own.

In [ ]:
import pandas as pd

# Example: load from Unity Catalog inference table
# df = spark.table("catalog.schema.inference_table").toPandas()

# Or use a small in-memory example for demo
df = pd.DataFrame({
    "request_id": ["a", "b"],
    "prompt": ["What is 2+2?", "Capital of France?"],
    "response": ["4", "Paris"],
})

with mlflow.start_run(run_name="eval_batch"):
    mlflow.log_param("num_rows", len(df))
    mlflow.log_table(df, "inference_sample.json")
    # Add custom metrics (e.g. from an eval set or scorer)
    mlflow.log_metric("sample_metric", 1.0)

print("Logged", len(df), "rows to run. View in MLflow UI.")

## 5. Multimodal / thinking models

For models that return **thinking** or **multimodal** content (e.g. Claude, reasoning traces), the same pattern applies: call the model, use `@mlflow.trace`, and log the run. Inputs/outputs and token usage appear in the trace. Use **GENAI evaluation** in MLflow to add scorers (relevance, correctness, etc.) on top of your traces.

In [ ]:
# Example: log a simple run that could represent a thinking-model response
with mlflow.start_run(run_name="thinking_example"):
    mlflow.log_param("model_type", "thinking")
    mlflow.log_param("prompt", "Explain step by step: 3 * 4")
    mlflow.log_metric("steps", 3)
print("Use MLflow UI to inspect traces and add GENAI evaluators.")